# Module 4 SHAP analysis

This notebook provides the assignment-facing SHAP analysis for the selected Module 4 model. The reproducible implementation lives in `src/models/explain_module4_random_forest.py`; the notebook runs that analysis and presents the resulting global and local explanations in one place.

The selected model is the tuned 12-feature Random Forest with isotonic calibration. SHAP is applied to the base Random Forest because the isotonic layer rescales probabilities after the model has produced its score. Raw and calibrated probabilities are therefore shown separately in the local explanations.


## Modelling chronology

- Random Forest training: through 6 July 2016
- Isotonic calibration: 7–13 July 2016
- Explanation population: final holdout, 14–23 July 2016

The final holdout is used for explanation only. It is not used to tune the model or select hyperparameters.


In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
from IPython.display import Image, display

cwd = Path.cwd().resolve()
if (cwd / 'src').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'src').exists():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError('Run this notebook from the repository root or notebooks directory.')

PROJECT_ROOT


## Reproduce the SHAP analysis

The next cell reruns the committed SHAP workflow. This keeps the notebook short while preserving the full modelling logic in a normal Python module that is easier to test and version-control.


In [ ]:
script = PROJECT_ROOT / 'src/models/explain_module4_random_forest.py'
subprocess.run([sys.executable, str(script)], cwd=PROJECT_ROOT, check=True)


## Global explanation

Mean absolute SHAP values show which predictors have the greatest influence on the Random Forest output across the holdout sample. The analysis is descriptive of model behaviour; it does not establish causal effects.


In [ ]:
global_path = PROJECT_ROOT / 'reports/tables/module4_shap_global_importance.csv'
global_importance = pd.read_csv(global_path)
global_importance.head(12)


In [ ]:
display(Image(filename=str(PROJECT_ROOT / 'reports/figures/module4/shap_global_importance.png')))
display(Image(filename=str(PROJECT_ROOT / 'reports/figures/module4/shap_global_beeswarm.png')))


The strongest global signals are concentrated in recharge behaviour, especially `cnt_ma_rech90`, `sumamnt_ma_rech90`, `sumamnt_ma_rech30`, `cnt_ma_rech30`, and `daily_decr30`. This is consistent with the broader feature-selection and sensitivity work: recharge behaviour carries most of the stable predictive signal, while several additional variables contribute more unevenly across time.


## Local explanations

Three representative holdout cases were selected near the 10th, 50th, and 90th percentiles of calibrated risk. The table below lists the SHAP contribution of every feature for each case.


In [ ]:
local_path = PROJECT_ROOT / 'reports/tables/module4_shap_local_cases.csv'
local_cases = pd.read_csv(local_path)

for case in ['low_risk', 'typical_risk', 'high_risk']:
    print(f'\n{case.replace("_", " ").title()}')
    display(
        local_cases.loc[local_cases['case'] == case]
        .sort_values('importance_rank')
        [['feature', 'feature_value', 'shap_value', 'direction']]
        .head(5)
    )


In [ ]:
for name in ['low_risk', 'typical_risk', 'high_risk']:
    print(name.replace('_', ' ').title())
    display(Image(filename=str(PROJECT_ROOT / f'reports/figures/module4/shap_local_{name}.png')))


The local explanations reinforce the global pattern. Higher recharge frequency and recharge amount often push lower-risk observations downward, while `rental30` and certain recharge-amount patterns can push higher-risk observations upward. These are explanations of the fitted model's internal logic, not recommendations to customers and not evidence that changing a feature would cause repayment behaviour to change.


## Interpretation and limits

The SHAP results add confidence that the selected model is using signals that are consistent with the broader modelling work rather than relying on a single opaque variable. They also expose an important limitation: some of the strongest predictors, particularly the decrement and recharge variables, show temporal movement elsewhere in the project. Explainability therefore supports understanding of the model but does not remove the need for temporal monitoring.

A separate constrained counterfactual analysis is available in `reports/tables/module4_counterfactual_summary.json` and `reports/figures/module4/counterfactual_risk_changes.png`.


## Supporting repository evidence

- `src/models/explain_module4_random_forest.py`
- `reports/tables/module4_shap_global_importance.csv`
- `reports/tables/module4_shap_local_cases.csv`
- `reports/tables/module4_shap_summary.json`
- `docs/model-documentation/module4_experiment_record.md`
- `docs/model-documentation/model_card.md`
